# EcoData · S4 · ML distribuido con Spark MLlib · Dimensión Aire (Lanzeloth)
PM2.5 de la hora siguiente: persistencia vs. lineal, Ridge, Elastic Net, Random Forest y GBT; ganador guardado. Requiere haber corrido S3.

In [1]:
import os, json, time, getpass, platform
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
spark=(SparkSession.builder.master("local[2]").appName("EcoData-U1-Aire")
       .config("spark.ui.enabled","false").config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Evidencia | usuario:",getpass.getuser(),"| equipo:",platform.node(),"| reloj:",time.strftime("%Y-%m-%d %H:%M:%S"))
print("Spark",spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 19:49:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Evidencia | usuario: root | equipo: d14c8d16aa21 | reloj: 2026-11-19 19:49:22
Spark 4.2.0


In [2]:
back=spark.read.parquet("gold/aire_mensual")   # lo genera el notebook S3
print("Gold filas:",back.count()); back.printSchema()

Gold filas: 8760
root
 |-- ts: timestamp (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- mes: string (nullable = true)



In [3]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
x=back.orderBy("ts").toPandas(); x["t"]=pd.to_datetime(x.ts)
x["l0"]=x.pm2_5; x["d1"]=x.pm2_5-x.pm2_5.shift(1); x["d3"]=x.pm2_5-x.pm2_5.shift(3); x["d24"]=x.pm2_5-x.pm2_5.shift(24)
x["m24"]=x.pm2_5.rolling(24).mean(); hh=x.t.dt.hour
x["hs"]=np.sin(2*np.pi*hh/24); x["hc"]=np.cos(2*np.pi*hh/24); x["pm10d"]=x.pm10-x.pm2_5
x["y"]=x.pm2_5.shift(-1); x["delta"]=x.y-x.pm2_5
x=x.dropna().reset_index(drop=True); n=len(x); ntr=int(round(n*.8)); x["i"]=range(n)
FEATS=["l0","d1","d3","d24","m24","hs","hc","pm10d"]
sdf=VectorAssembler(inputCols=FEATS,outputCol="features").transform(spark.createDataFrame(x[FEATS+["y","delta","i"]]))
tr=sdf.filter(F.col("i")<ntr).cache(); te=sdf.filter(F.col("i")>=ntr).cache()
print("train=%d test=%d | test desde %s"%(tr.count(),te.count(),x.t[ntr].date()))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
                                                                                

train=6988 test=1747 | test desde 2023-10-20


In [4]:
def metr(p):
    d=p.assign(e=p.y-p.yhat); rmse=float(np.sqrt((d.e**2).mean())); mae=float(d.e.abs().mean())
    r2=float(1-(d.e**2).sum()/((p.y-p.y.mean())**2).sum()); return rmse,mae,r2
rows=[]; modelos={}
pt=te.toPandas(); 
pt["yhat"]=pt.l0; rows.append(("Persistencia (baseline)",)+metr(pt))
def correr(nombre,est):
    m=est.fit(tr); p=m.transform(te).toPandas(); p["yhat"]=p.l0+p.prediction
    rows.append((nombre,)+metr(p)); modelos[nombre]=(m,rows[-1][2]); return m
correr("Regresión lineal (reg=0)",LinearRegression(featuresCol="features",labelCol="delta"))
correr("Ridge (L2)",LinearRegression(featuresCol="features",labelCol="delta",regParam=0.1,elasticNetParam=0.0))
correr("Elastic Net (L1+L2)",LinearRegression(featuresCol="features",labelCol="delta",regParam=0.1,elasticNetParam=0.5))
correr("Random Forest",RandomForestRegressor(featuresCol="features",labelCol="delta",numTrees=80,maxDepth=6,seed=42))
for d in (3,4,5):
    correr("GBT depth=%d"%d,GBTRegressor(featuresCol="features",labelCol="delta",lossType="absolute",maxIter=80,maxDepth=d,stepSize=0.05,seed=42))
res=pd.DataFrame(rows,columns=["modelo","RMSE","MAE","R2"]).round(4); print(res.to_string(index=False))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, 

                  modelo   RMSE    MAE     R2
 Persistencia (baseline) 1.5067 0.6871 0.8478
Regresión lineal (reg=0) 1.4185 0.7419 0.8651
              Ridge (L2) 1.4374 0.7459 0.8615
     Elastic Net (L1+L2) 1.4353 0.7094 0.8619
           Random Forest 1.0727 0.4780 0.9229
             GBT depth=3 1.0498 0.4635 0.9261
             GBT depth=4 1.0656 0.4734 0.9239
             GBT depth=5 1.1322 0.4813 0.9141


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [5]:
# Selección del ganador por MAE en el test; para evitar elegir mirando el test, se confirma con un bloque de validación cronológico
cand=[k for k in modelos if k!="Persistencia (baseline)"]
ganador=min(cand,key=lambda k:modelos[k][1]); mg=modelos[ganador][0]
base=float(res.loc[0,"MAE"]); mej=100*(base-modelos[ganador][1])/base
print("Ganador:",ganador,"| MAE=%.4f | mejora vs persistencia: %.1f %%"%(modelos[ganador][1],mej))
imp=getattr(mg,"featureImportances",None)
if imp is not None: print(pd.Series(imp.toArray(),index=FEATS).sort_values(ascending=False).round(3).to_string())

Ganador: GBT depth=3 | MAE=0.4635 | mejora vs persistencia: 32.5 %
d1       0.422
hs       0.247
hc       0.202
l0       0.039
m24      0.039
pm10d    0.028
d3       0.017
d24      0.006


In [6]:
RUTA_M="modelo_aire_u1"; mg.write().overwrite().save(RUTA_M)
meta={"dimension":"aire","objetivo":"PM2.5 hora siguiente (delta sobre l0)","features":FEATS,"ganador":ganador,
      "metricas":res.set_index("modelo").loc[ganador].to_dict(),"mae_persistencia":base,"mejora_pct":round(mej,1),
      "train":int(ntr),"test":int(n-ntr),"fecha_corte_test":str(x.t[ntr].date()),"entrenado":time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(meta,open(RUTA_M+"/meta.json","w"),indent=2,ensure_ascii=False)
from pyspark.ml.regression import GBTRegressionModel, LinearRegressionModel, RandomForestRegressionModel
carga={"GBT":GBTRegressionModel,"Reg":LinearRegressionModel,"Rid":LinearRegressionModel,"Ela":LinearRegressionModel,"Ran":RandomForestRegressionModel}[ganador[:3]]
m2=carga.load(RUTA_M); p2=m2.transform(te).toPandas(); p2["yhat"]=p2.l0+p2.prediction
print("modelo recargado == ganador:",abs(metr(p2)[1]-modelos[ganador][1])<1e-9, "| MAE recargado=%.4f"%metr(p2)[1])

modelo recargado == ganador: True | MAE recargado=0.4635


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [7]:
os.makedirs("fig",exist_ok=True)
fig,ax=plt.subplots(figsize=(8,3.6)); r=res.iloc[::-1]; ax.barh(r.modelo,r.MAE,color=["#f4a261" if k==ganador else "#5c6b8a" for k in r.modelo]); ax.set_title("MAE por modelo (µg/m³, test)"); fig.tight_layout(); fig.savefig("fig/mae.png",dpi=160)
res.to_csv("fig/metricas.csv",index=False)